# 3) MTG Feature Pipline
- Running extracted audio files though the PSIC Features Pipline
- Once created, we can use these features to pseudolabel from our previously saved model

## Library Imports

In [ ]:
import librosa
import os
import zipfile
import numpy as np
import pandas as pd
import math
from sklearn.decomposition import PCA
import time
import subprocess

## Pulling file
- Use whichever size was pulled from Jamendo

In [ ]:
# Path to the zip file
zip_path = "jamendo_balanced_.zip"  # Whatever is named from MTG extraction

# Directory to extract to
extract_dir = "jamendo_balanced_"

# Create the directory if it doesn't exist
os.makedirs(extract_dir, exist_ok=True)

# Extract the zip file
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)
    
# Now access the manifest file
manifest_path = os.path.join(extract_dir, "manifest_balanced.csv")

# Read the CSV file
m = pd.read_csv(manifest_path)
names = m["file"].tolist()   # e.g., "01_106222.mp3", ...
names[:5]

In [ ]:
# Jamendo_
names_ = m["file"].head(200).tolist() # Names of the total count of songs extracted
print(len(names_), names_[:40])

## Audio Pipeline

In [ ]:
def music_analysis(names, path):
    all_list=['id','tempo','mfccs_mean','mfccs_std','chroma_mean','spectral_centroid','spectral_bandwidth','spectral_contrast','spectral_flatness','spectral_rolloff','tonnetz']
    all_dataframe = pd.DataFrame(columns=all_list)
    # Creates empty table
    mfcc_all = pd.DataFrame()
    chroma_all = pd.DataFrame()
    cent_all = pd.DataFrame()
    bw_all = pd.DataFrame() 
    contrast_all = pd.DataFrame()
    flat_all = pd.DataFrame()
    rolloff_all = pd.DataFrame()
    tonnetz_all = pd.DataFrame()    
    a=-1
    rows = []
    for i, name in enumerate(names):
        print(i+1, ": ",  name)
        file_path = os.path.join(path, name)
        y,sr=librosa.load(file_path, sr = None, duration = 180 )
        each_one = []
        a = a+1
        # tempo
        onset_env = librosa.onset.onset_strength(y=y, sr=sr)
        tempo = librosa.beat.tempo(onset_envelope=onset_env, sr=sr)
        tempo = float(tempo[0])
        print (f"tempo = {tempo:.0f}")
        print("----------------")
        y,sr=librosa.load(file_path, sr = 2205, duration = 180 )
        # MFCCs
        mfccs = librosa.feature.mfcc(y=y, sr=sr)
        mfccs_mean = np.average(mfccs)
        mfccs_std = np.std(mfccs)
        #Chroma
        chroma = librosa.feature.chroma_stft(y=y, sr=sr)
        chroma_mean = np.average(chroma)
        #spectral_centroid
        cent = librosa.feature.spectral_centroid(y=y, sr=sr)
        cent_mean = np.average(cent)
        #spectral_bandwidth
        bw = librosa.feature.spectral_bandwidth(y=y, sr=sr)
        bw_mean = np.average(bw)
        #spectral_contrast
        S = np.abs(librosa.stft(y))
        contrast = librosa.feature.spectral_contrast(S=S, sr=22050)
        contrast_mean = np.average(contrast)
        #spectral_flatness
        flat = librosa.feature.spectral_flatness(y=y)
        flat_mean = np.average(flat)
        #spectral_rolloff
        rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)
        rolloff_mean = np.average(rolloff)
        #tonnetz
        tonnetz = librosa.feature.tonnetz(y=y, sr=22050)
        tonnetz_mean = np.average(tonnetz)
        
        # This place only merges single-element values such as the mean and variance.
        rows.append([a, tempo, mfccs_mean, mfccs_std, chroma_mean, cent_mean, bw_mean,
             contrast_mean, flat_mean, rolloff_mean, tonnetz_mean])

        # Here, the MFCCs are merged for subsequent PCA.
        l = len(mfccs[0])
        n = 776/l
        if (n <= 1): # This is used to pad/complete songs that are shorter than 3 minutes.
            mfccs = mfccs
        else:
            m = math.ceil(n)-1
            mfccs_1 = mfccs
            for i in range(m):
                mfccs_1 = np.hstack((mfccs_1,mfccs)) 
            mfccs = mfccs_1[:,0:776]
        mfccs = mfccs.reshape(1, 15520)
        mfcc_one = pd.DataFrame(mfccs)
        mfcc_all = pd.concat([mfcc_all, mfcc_one], ignore_index=True)
        
        # merge chroma features
        l = len(chroma[0])
        n = 776/l
        if (n <= 1): # For songs shorter than 3 minutes.
            chroma = chroma
        else:
            m = math.ceil(n)-1
            chroma_1 = chroma
            for i in range(m):
                chroma_1 = np.hstack((chroma_1,chroma)) 
            chroma = chroma_1[:,0:776]
        chroma = chroma.reshape(1, 9312)
        chroma_one = pd.DataFrame(chroma)
        chroma_all = pd.concat([chroma_all, chroma_one], ignore_index=True)        

        # merge cent features
        l = len(cent[0])
        n = 776/l
        if (n <= 1): # songs shorter than 3 minutes
            cent = cent
        else:
            m = math.ceil(n)-1
            cent_1 = cent
            for i in range(m):
                cent_1 = np.hstack((cent_1,cent)) 
            cent = cent_1[:,0:776]
        cent_one = pd.DataFrame(cent)
        cent_all = pd.concat([cent_all, cent_one], ignore_index=True)
 
        # merge spectral bandwidth features
        l = len(bw[0])
        n = 776/l
        if (n <= 1): # songs < 3 minutes
            bw = bw
        else:
            m = math.ceil(n)-1
            bw_1 = bw
            for i in range(m):
                bw_1 = np.hstack((bw_1,bw)) 
            bw = bw_1[:,0:776]
        bw_one = pd.DataFrame(bw)
        bw_all = pd.concat([bw_all, bw_one], ignore_index=True)
        
        # merge spectral contrast features
        l = len(contrast[0])
        n = 776/l
        if (n <= 1): # songs < 3 minutes
            contrast = contrast
        else:
            m = math.ceil(n)-1
            contrast_1 = contrast
            for i in range(m):
                contrast_1 = np.hstack((contrast_1,contrast)) 
            contrast = contrast_1[:,0:776]
        contrast = contrast.reshape(1, 5432)
        contrast_one = pd.DataFrame(contrast)
        contrast_all = pd.concat([contrast_all, contrast_one], ignore_index=True)
        
        # merge the spectral flatness features
        l = len(flat[0])
        n = 776/l
        if (n == 1): # songs < 3 minutes
            flat = flat
        else:
            m = math.ceil(n)-1
            flat_1 = flat
            for i in range(m):
                flat_1 = np.hstack((flat_1,flat)) 
            flat = flat_1[:,0:776]
        flat_one = pd.DataFrame(flat)
        flat_all = pd.concat([flat_all, flat_one], ignore_index=True)
        
        # merge rolloff features
        l = len(rolloff[0])
        n = 776/l
        if (n <= 1): # songs < 3 minutes
            rolloff = rolloff
        else:
            m = math.ceil(n)-1
            rolloff_1 = rolloff
            for i in range(m):
                rolloff_1 = np.hstack((rolloff_1,rolloff)) 
            rolloff = rolloff_1[:,0:776]
        rolloff_one = pd.DataFrame(rolloff)
        rolloff_all = pd.concat([rolloff_all, rolloff_one], ignore_index=True)
        
        # merge the tonal centroid/Tonnetz
        l = len(tonnetz[0])
        n = 776/l
        if (n <= 1): # songs < 3 minutes
            tonnetz = tonnetz
        else:
            m = math.ceil(n)-1
            tonnetz_1 = tonnetz
            for i in range(m):
                tonnetz_1 = np.hstack((tonnetz_1,tonnetz)) 
            tonnetz = tonnetz_1[:,0:776]
        tonnetz = tonnetz.reshape(1, 4656)
        tonnetz_one = pd.DataFrame(tonnetz)
        tonnetz_all = pd.concat([tonnetz_all, tonnetz_one], ignore_index=True)
        
    # save to single element table
    all_dataframe = pd.DataFrame(rows, columns=all_list)
    all_dataframe.to_csv('mid_level_feature_1.csv',header=None, index=None)
       
    # saving raw data，to avoid rerunning
    mfcc_all.to_csv('mid_level_mfccs_all.csv')
    chroma_all.to_csv('mid_level_chroma_all.csv')
    cent_all.to_csv('mid_level_cent_all.csv') 
    bw_all.to_csv('mid_level_bw_all.csv')
    contrast_all.to_csv('mid_level_contrast_all.csv')
    flat_all.to_csv('mid_level_flat_all.csv')
    rolloff_all.to_csv('mid_level_rolloff_all.csv')
    tonnetz_all.to_csv('mid_level_tonnetz_all.csv')
        
    #Perform PCA dimensionality reduction on the MFCCs    
    n=50 #50
    pca = PCA(n_components=n)
    pca.fit(mfcc_all.values)
    mfcc_reduc = pca.transform(mfcc_all)
    mfcc_reduc_dataframe = pd.DataFrame(mfcc_reduc)
    mfcc_reduc_dataframe.to_csv('mid_levle_mfccs_reduc.csv') # used for model
    mfcc_variance = pca.explained_variance_ratio_
    mfcc_reduc_1 = pd.DataFrame(mfcc_variance)
    mfcc_sum = mfcc_reduc_1.sum()
    print('mid_level_mfcc_variance', mfcc_sum)
    mfcc_sum.to_csv('mid_level_mfcc_variance.csv')
    
    #chroma PCA
    n=50 #50
    pca = PCA(n_components=n)
    pca.fit(chroma_all.values)
    chroma_reduc = pca.transform(chroma_all)
    chroma_reduc_dataframe = pd.DataFrame(chroma_reduc)
    chroma_reduc_dataframe.to_csv('mid_levle_chroma_reduc.csv') # used for model
    chroma_variance = pca.explained_variance_ratio_
    chroma_reduc_1 = pd.DataFrame(chroma_variance)
    chroma_sum = chroma_reduc_1.sum()
    print('mid_level_chroma_variance', chroma_sum)
    chroma_sum.to_csv('mid_level_chroma_variance.csv')
    
    #cent PCA
    n=50 #50
    pca = PCA(n_components=n)
    pca.fit(cent_all.values)
    cent_reduc = pca.transform(cent_all)
    cent_reduc_dataframe = pd.DataFrame(cent_reduc)
    cent_reduc_dataframe.to_csv('mid_levle_cent_reduc.csv') # used for model
    cent_variance = pca.explained_variance_ratio_
    cent_reduc_1 = pd.DataFrame(cent_variance)
    cent_sum = cent_reduc_1.sum()
    print('mid_level_cent_variance', cent_sum)
    cent_sum.to_csv('mid_level_cent_variance.csv')   

    #bw PCA
    n=50 #50
    pca = PCA(n_components=n)
    pca.fit(bw_all.values)
    bw_reduc = pca.transform(bw_all)
    bw_reduc_dataframe = pd.DataFrame(bw_reduc)
    bw_reduc_dataframe.to_csv('mid_levle_bw_reduc.csv') # used for model
    bw_variance = pca.explained_variance_ratio_
    bw_reduc_1 = pd.DataFrame(bw_variance)
    bw_sum = bw_reduc_1.sum()
    print('mid_level_bw_variance', bw_sum)
    bw_sum.to_csv('mid_level_bw_variance.csv')  
    
    #contrast PCA
    n=50 #50
    pca = PCA(n_components=n)
    pca.fit(contrast_all.values)
    contrast_reduc = pca.transform(contrast_all)
    contrast_reduc_dataframe = pd.DataFrame(contrast_reduc)
    contrast_reduc_dataframe.to_csv('mid_levle_contrast_reduc.csv') # used for model
    contrast_variance = pca.explained_variance_ratio_
    contrast_reduc_1 = pd.DataFrame(contrast_variance)
    contrast_sum = contrast_reduc_1.sum()
    print('mid_level_contrast_variance', contrast_sum)
    contrast_sum.to_csv('mid_level_contrast_variance.csv')
    
    #flat PCA
    n=50 #50
    pca = PCA(n_components=n)
    pca.fit(flat_all.values)
    flat_reduc = pca.transform(flat_all)
    flat_reduc_dataframe = pd.DataFrame(flat_reduc)
    flat_reduc_dataframe.to_csv('mid_levle_flat_reduc.csv') # used for model
    flat_variance = pca.explained_variance_ratio_
    flat_reduc_1 = pd.DataFrame(flat_variance)
    flat_sum = flat_reduc_1.sum()
    print('mid_level_flat_variance', flat_sum)
    flat_sum.to_csv('mid_level_flat_variance.csv') 
    
    #rolloff PCA
    n=50 #50
    pca = PCA(n_components=n)
    pca.fit(rolloff_all.values)
    rolloff_reduc = pca.transform(rolloff_all)
    rolloff_reduc_dataframe = pd.DataFrame(rolloff_reduc)
    rolloff_reduc_dataframe.to_csv('mid_levle_rolloff_reduc.csv') # used for model
    rolloff_variance = pca.explained_variance_ratio_
    rolloff_reduc_1 = pd.DataFrame(rolloff_variance)
    rolloff_sum = rolloff_reduc_1.sum()
    print('mid_level_rolloff_variance', rolloff_sum)
    rolloff_sum.to_csv('mid_level_rolloff_variance.csv')  
    
    #tonnetz PCA
    n=50 #50
    pca = PCA(n_components=n)
    pca.fit(tonnetz_all.values)
    tonnetz_reduc = pca.transform(tonnetz_all)
    tonnetz_reduc_dataframe = pd.DataFrame(tonnetz_reduc)
    tonnetz_reduc_dataframe.to_csv('mid_levle_tonnetz_reduc.csv') # used for model
    tonnetz_variance = pca.explained_variance_ratio_
    tonnetz_reduc_1 = pd.DataFrame(tonnetz_variance)
    tonnetz_sum = tonnetz_reduc_1.sum()
    print('mid_level_tonnetz_variance', tonnetz_sum)
    tonnetz_sum.to_csv('mid_level_tonnetz_variance.csv')
    
    
    return all_dataframe

In [ ]:
import joblib
import os

os.makedirs("models", exist_ok=True)

# Save fitted PCA objects
joblib.dump(pca_mfcc, "models/pca_mfcc.joblib")
joblib.dump(pca_chroma, "models/pca_chroma.joblib")
joblib.dump(pca_cent, "models/pca_cent.joblib")
joblib.dump(pca_bw, "models/pca_bw.joblib")
joblib.dump(pca_contrast, "models/pca_contrast.joblib")
joblib.dump(pca_flat, "models/pca_flat.joblib")
joblib.dump(pca_rolloff, "models/pca_rolloff.joblib")
joblib.dump(pca_tonnetz, "models/pca_tonnetz.joblib")

print("PCA artifacts saved.")

## Running Pipeline

In [ ]:
# Time
start_time = time.time()

# Running Pipeline
all_final_ = music_analysis(names_, path = extract_dir)

print(all_final_.shape)
print(all_final_.head())

# Calculate elapsed time
elapsed_time = time.time() - start_time
hours, remainder = divmod(elapsed_time, 3600)
minutes, seconds = divmod(remainder, 60)

print(f"Time elapsed: {int(hours)}h {int(minutes)}m {seconds:.2f}s")

### Combining Dataset

In [11]:
# --- helper: load a PCA block + rename columns to "<base> 1..50" ---
def load_block(filename, base_name):
    df = pd.read_csv(f"{filename}", index_col=0).reset_index(drop=True)
    df.columns = [f"{base_name} {i+1}" for i in range(df.shape[1])]  # 1-indexed, spaces
    return df

# 1) tempo (scalar file saved with header=None)
scalars_raw = pd.read_csv(f"mid_level_feature_1.csv", header=None)
tempo = scalars_raw.iloc[:, [1]].rename(columns={1: "tempo"}).reset_index(drop=True)

# 2) reduced PCA blocks (rename to match training schema text)
bw     = load_block("mid_levle_bw_reduc.csv",       "spectral bandwidth")
cent   = load_block("mid_levle_cent_reduc.csv",     "spectral centroid")
chroma = load_block("mid_levle_chroma_reduc.csv",   "chromagram")
contr  = load_block("mid_levle_contrast_reduc.csv", "spectral contrast")
flat   = load_block("mid_levle_flat_reduc.csv",     "spectral flatness")
mfcc   = load_block("mid_levle_mfccs_reduc.csv",    "MFCCs")
roll   = load_block("mid_levle_rolloff_reduc.csv",  "spectral rolloff")
ton    = load_block("mid_levle_tonnetz_reduc.csv",  "tonnetz")

# 3) build X (tempo last, as you want)
blocks = [bw, cent, chroma, contr, flat, mfcc, roll, ton]
n = len(tempo)
assert all(len(b) == n for b in blocks), "Row mismatch between tempo and PCA blocks"

X = pd.concat(blocks + [tempo], axis=1)
print("X shape before ordering:", X.shape)  # expect (n_songs, 401)


# 5) save
X.to_csv("X_features_401.csv", index=False)
print("Saved X_features_401.csv with shape:", X.shape)

FileNotFoundError: [Errno 2] No such file or directory: 'mid_level_feature_1.csv'

We now have our featured dataset for pseudolabelling

In [25]:
# =========================================
# Reload Raw Features + Save PCA Artifacts
# =========================================

import pandas as pd

mfcc_all     = pd.read_csv("PSIC_Extras/mid_level_mfccs_all.csv", index_col=0)
chroma_all   = pd.read_csv("PSIC_Extras/mid_level_chroma_all.csv", index_col=0)
contrast_all = pd.read_csv("PSIC_Extras/mid_level_contrast_all.csv", index_col=0)
cent_all     = pd.read_csv("PSIC_Extras/mid_level_cent_all.csv", index_col=0)
bw_all       = pd.read_csv("PSIC_Extras/mid_level_bw_all.csv", index_col=0)
flat_all     = pd.read_csv("PSIC_Extras/mid_level_flat_all.csv", index_col=0)
rolloff_all  = pd.read_csv("PSIC_Extras/mid_level_rolloff_all.csv", index_col=0)
tonnetz_all  = pd.read_csv("PSIC_Extras/mid_level_tonnetz_all.csv", index_col=0)

print(mfcc_all.shape)
print(cent_all.shape)

(200, 15520)
(200, 776)


In [18]:
# =====================================
# Add Tempo Feature
# =====================================

# This file should contain scalar features from your extraction pipeline
# column 0 = id, column 1 = tempo, then other summary features
scalars_raw = pd.read_csv("PSIC_Extras/mid_level_feature_1.csv", header=None)

# tempo is column 1
tempo = scalars_raw.iloc[:, [1]].reset_index(drop=True)
tempo.columns = ["tempo"]

print("tempo shape:", tempo.shape)
print(tempo.head())

tempo shape: (200, 1)
        tempo
0  139.674831
1  114.843750
2  132.512019
3  120.185320
4  114.843750


In [26]:
from sklearn.decomposition import PCA

# Recreate PCA objects
pca_mfcc = PCA(n_components=50)
pca_chroma = PCA(n_components=50)
pca_contrast = PCA(n_components=50)
pca_cent = PCA(n_components=50)
pca_bw = PCA(n_components=50)
pca_flat = PCA(n_components=50)
pca_rolloff = PCA(n_components=50)
pca_tonnetz = PCA(n_components=50)

# Fit PCA objects ONLY
pca_mfcc.fit(mfcc_all)
pca_chroma.fit(chroma_all)
pca_contrast.fit(contrast_all)
pca_cent.fit(cent_all)
pca_bw.fit(bw_all)
pca_flat.fit(flat_all)
pca_rolloff.fit(rolloff_all)
pca_tonnetz.fit(tonnetz_all)

print("All PCA objects refit.")

All PCA objects refit.


In [27]:
import joblib
import os

os.makedirs("models", exist_ok=True)

joblib.dump(pca_mfcc, "models/pca_mfcc.joblib")
joblib.dump(pca_chroma, "models/pca_chroma.joblib")
joblib.dump(pca_contrast, "models/pca_contrast.joblib")
joblib.dump(pca_cent, "models/pca_cent.joblib")
joblib.dump(pca_bw, "models/pca_bw.joblib")
joblib.dump(pca_flat, "models/pca_flat.joblib")
joblib.dump(pca_rolloff, "models/pca_rolloff.joblib")
joblib.dump(pca_tonnetz, "models/pca_tonnetz.joblib")

print("PCA artifacts saved.")

PCA artifacts saved.


In [28]:
# =====================================
# Rebuild Final 401-Feature Matrix
# =====================================

# Apply PCA transforms
mfcc_reduc = pca_mfcc.transform(mfcc_all)
chroma_reduc = pca_chroma.transform(chroma_all)
contrast_reduc = pca_contrast.transform(contrast_all)
cent_reduc = pca_cent.transform(cent_all)
bw_reduc = pca_bw.transform(bw_all)
flat_reduc = pca_flat.transform(flat_all)
rolloff_reduc = pca_rolloff.transform(rolloff_all)
tonnetz_reduc = pca_tonnetz.transform(tonnetz_all)

# Convert to DataFrames
import pandas as pd

mfcc_df = pd.DataFrame(mfcc_reduc)
chroma_df = pd.DataFrame(chroma_reduc)
contrast_df = pd.DataFrame(contrast_reduc)
cent_df = pd.DataFrame(cent_reduc)
bw_df = pd.DataFrame(bw_reduc)
flat_df = pd.DataFrame(flat_reduc)
rolloff_df = pd.DataFrame(rolloff_reduc)
tonnetz_df = pd.DataFrame(tonnetz_reduc)

# Rename columns for consistency
mfcc_df.columns = [f"MFCCs {i+1}" for i in range(mfcc_df.shape[1])]
chroma_df.columns = [f"chromagram {i+1}" for i in range(chroma_df.shape[1])]
contrast_df.columns = [f"spectral contrast {i+1}" for i in range(contrast_df.shape[1])]
cent_df.columns = [f"spectral centroid {i+1}" for i in range(cent_df.shape[1])]
bw_df.columns = [f"spectral bandwidth {i+1}" for i in range(bw_df.shape[1])]
flat_df.columns = [f"spectral flatness {i+1}" for i in range(flat_df.shape[1])]
rolloff_df.columns = [f"spectral rolloff {i+1}" for i in range(rolloff_df.shape[1])]
tonnetz_df.columns = [f"tonnetz {i+1}" for i in range(tonnetz_df.shape[1])]

# Combine all blocks
X = pd.concat([
    bw_df,
    cent_df,
    chroma_df,
    contrast_df,
    flat_df,
    mfcc_df,
    rolloff_df,
    tonnetz_df,
    tempo
], axis=1)

print("X shape:", X.shape)

X shape: (200, 401)


In [29]:
import joblib

joblib.dump(list(X.columns), "models/feature_columns.joblib")

print("Feature column ordering saved.")

Feature column ordering saved.
